In [1]:
!pip install opencv-python tensorflow keras numpy

In [2]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import AveragePooling2D, Dense, Dropout, Flatten
from tensorflow.keras.models import Model

# Load MobileNetV2 without the top layer
base_model = MobileNetV2(
    weights="imagenet", include_top=False, input_shape=(224, 224, 3)
)

# Construct the custom head for our mask binary classification
head_model = base_model.output
head_model = AveragePooling2D(pool_size=(7, 7))(head_model)
head_model = Flatten(name="flatten")(head_model)
head_model = Dense(128, activation="relu")(head_model)
head_model = Dropout(0.5)(head_model)
head_model = Dense(2, activation="softmax")(head_model)

# Place the head on top of the base model
model = Model(inputs=base_model.input, outputs=head_model)

# Save it locally so the video script can read it
model.save("mask_detector.h5")
print("Model placeholder 'mask_detector.h5' successfully created!")

/Users/bhumikachauhan/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


Model placeholder 'mask_detector.h5' successfully created!


In [3]:
import os
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array

# 1. Load the pre-trained Keras model
print("[INFO] Loading mask detector model...")
model = load_model("mask_detector.h5")

# 2. Load OpenCV's Haar Cascade for Face Detection
# Using cv2.data.haarcascades automatically finds the path on your local machine
cascade_path = os.path.join(
    cv2.data.haarcascades, "haarcascade_frontalface_default.xml"
)
face_cascade = cv2.CascadeClassifier(cascade_path)

# 3. Initialize Webcam Stream
print("[INFO] Starting video stream (Press 'q' to exit)...")
vs = cv2.VideoCapture(0)

while True:
    # Grab the current frame from the webcam
    ret, frame = vs.read()
    if not ret:
        break

    # Flip horizontally for a mirror effect, then get dimensions
    frame = cv2.flip(frame, 1)
    (h, w) = frame.shape[:2]

    # Convert to grayscale for Haar Cascade face detection
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect faces in the frame
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60)
    )

    # Process each detected face
    for x, y, face_w, face_h in faces:
        # Extract the Face Region of Interest (ROI)
        face_roi = frame[y : y + face_h, x : x + face_w]

        # Preprocess the face ROI to match Keras input expectations
        face_rgb = cv2.cvtColor(face_roi, cv2.COLOR_BGR2RGB)
        face_resized = cv2.resize(face_rgb, (224, 224))
        face_array = img_to_array(face_resized)
        # Scale pixels to [-1, 1] (Standard MobileNetV2 preprocessing)
        face_array = (face_array / 127.5) - 1.0
        face_batch = np.expand_dims(face_array, axis=0)

        # Predict using Keras model
        # Output structure: [prob_of_mask, prob_of_no_mask]
        (mask, withoutMask) = model.predict(face_batch, verbose=0)[0]

        # Determine class label and color for bounding box
        if mask > withoutMask:
            label = f"Mask: {mask * 100:.1f}%"
            color = (0, 255, 0)  # Green
        else:
            label = f"No Mask: {withoutMask * 100:.1f}%"
            color = (0, 0, 255)  # Red

        # Draw the bounding box and label text onto the video frame
        cv2.putText(
            frame,
            label,
            (x, y - 10),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.45,
            color,
            2,
        )
        cv2.rectangle(frame, (x, y), (x + face_w, y + face_h), color, 2)

    # Show the output stream in an external OpenCV window
    cv2.imshow("Real-Time Face Mask Detector", frame)

    # Break the loop when 'q' key is pressed
    key = cv2.waitKey(1) & 0xFF
    if key == ord("q"):
        break

# Clean up windows and release webcam hardware resource
vs.release()
cv2.destroyAllWindows()

[INFO] Loading mask detector model...


[INFO] Starting video stream (Press 'q' to exit)...
